This notebook generate a summary for each of the column in the input data set

Load libraries

In [ ]:
import chardet
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Loading Dataset and Initial Data Exploration

In [ ]:

file_path = '/content/drive/My Drive/Colab Notebooks/Ming_Cleaned_date_drop_Timestamp_LastResidance_LastBirthplace_Accept_Highestleague_0707.csv'
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nDataset info:")
df.info()

print("\nSample data:")
df.head()

Dataset shape: (13464, 50)

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13464 entries, 0 to 13463
Data columns (total 50 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   S. No.      13464 non-null  int64  
 1   GAD1        13464 non-null  int64  
 2   GAD2        13464 non-null  int64  
 3   GAD3        13464 non-null  int64  
 4   GAD4        13464 non-null  int64  
 5   GAD5        13464 non-null  int64  
 6   GAD6        13464 non-null  int64  
 7   GAD7        13464 non-null  int64  
 8   GADE        12815 non-null  object 
 9   SWL1        13464 non-null  int64  
 10  SWL2        13464 non-null  int64  
 11  SWL3        13464 non-null  int64  
 12  SWL4        13464 non-null  int64  
 13  SWL5        13464 non-null  int64  
 14  Game        13464 non-null  object 
 15  Platform    13464 non-null  object 
 16  Hours       13434 non-null  float64
 17  earnings    13464 non-null  object 
 18  whyplay     13464 non-nu

,S. No.,GAD1,GAD2,GAD3,GAD4,GAD5,GAD6,GAD7,GADE,SWL1,...,Age,Work,Degree,Birthplace,Residence,Reference,Playstyle,GAD_T,SWL_T,SPIN_T
0,1,0,0,0,0,1,0,0,Not difficult at all,3,...,25,Unemployed / between jobs,Bachelor (or equivalent),USA,USA,Reddit,Singleplayer,1,23,5.0
1,2,1,2,2,2,0,1,0,Somewhat difficult,3,...,41,Unemployed / between jobs,Bachelor (or equivalent),USA,USA,Reddit,Multiplayer - online - with strangers,8,16,33.0
2,3,0,2,2,0,0,3,1,Not difficult at all,2,...,32,Employed,Bachelor (or equivalent),Germany,Germany,Reddit,Singleplayer,8,17,31.0
3,4,0,0,0,0,0,0,0,Not difficult at all,2,...,28,Employed,Bachelor (or equivalent),USA,USA,Reddit,Multiplayer - online - with online acquaintanc...,0,17,11.0
4,5,2,1,2,2,2,3,2,Very difficult,2,...,19,Employed,High school diploma (or equivalent),USA,South Korea,Reddit,Multiplayer - online - with strangers,14,14,13.0


In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
S. No.,13464.0,7096.839201,4114.478220,1.0,3532.75,7087.5,10654.25,14250.0
GAD1,13464.0,0.860963,0.926542,0.0,0.00,1.0,1.00,3.0
GAD2,13464.0,0.673351,0.915724,0.0,0.00,0.0,1.00,3.0
GAD3,13464.0,0.965761,0.982776,0.0,0.00,1.0,2.00,3.0
GAD4,13464.0,0.724079,0.921971,0.0,0.00,0.0,1.00,3.0
GAD5,13464.0,0.488042,0.837014,0.0,0.00,0.0,1.00,3.0
GAD6,13464.0,0.911022,0.931168,0.0,0.00,1.0,1.00,3.0
GAD7,13464.0,0.588755,0.894408,0.0,0.00,0.0,1.00,3.0
SWL1,13464.0,3.720440,1.736264,1.0,2.00,4.0,5.00,7.0
SWL2,13464.0,4.602050,1.696276,1.0,3.00,5.0,6.00,7.0


Missing Values Analysis and Categorical Features Overview

In [ ]:
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0].sort_values(ascending=False))

missing_percentage = (missing_values / len(df)) * 100
print("\nPercentage of missing values:")
print(missing_percentage[missing_percentage > 0].sort_values(ascending=False))

categorical_columns = df.select_dtypes(include=['object']).columns
for col in categorical_columns:
    print(f"\nUnique values in {col}:")
    print(df[col].value_counts().head())
    print(f"Number of unique values: {df[col].nunique()}")

Missing values per column:
League        1852
Degree        1577
SPIN_T         650
GADE           649
SPIN11         187
SPIN13         187
SPIN17         175
SPIN12         168
SPIN5          166
SPIN10         160
SPIN4          159
SPIN9          158
SPIN14         156
SPIN6          156
SPIN2          154
SPIN15         147
SPIN16         147
SPIN8          144
SPIN3          140
SPIN7          138
SPIN1          124
streams        100
Work            38
Hours           30
Narcissism      23
Reference       15
dtype: int64

Percentage of missing values:
League        13.755199
Degree        11.712715
SPIN_T         4.827689
GADE           4.820261
SPIN11         1.388889
SPIN13         1.388889
SPIN17         1.299762
SPIN12         1.247772
SPIN5          1.232917
SPIN10         1.188354
SPIN4          1.180927
SPIN9          1.173500
SPIN14         1.158645
SPIN6          1.158645
SPIN2          1.143791
SPIN15         1.091800
SPIN16         1.091800
SPIN8          1.069519
SPI

In [ ]:
print("Count of individuals by age:")
display(df['Age'].value_counts().sort_index())

Count of individuals by age:


,count
Age,
18,3588
19,2159
20,1855
21,1507
22,1135
23,872
24,660
25,514
26,338


In [ ]:
pd.set_option('display.max_rows', None)
print("Count of individuals by hours:")
display(df['Hours'].value_counts().sort_index())

Count of individuals by hours:


,count
Hours,
0.0,12
1.0,22
2.0,74
3.0,105
4.0,143
5.0,380
6.0,199
7.0,237
8.0,386
